# 02 — Track B: final-impression validation (individual level, fMRI cohort)

**One sentiment, two ground truths.**  sentiment scores come from the fMRI
participants' spoken reflections (per participant x run x character). We validate:

| | ground truth | cohort | person resolution | time resolution | items |
|---|---|---|---|---|---|
| **Track A** (`01`) | SONA behavioral survey | *different* people, group link only | group (3 scramble groups) | run-resolved (10 runs) | 16 |
| **Track B** (this nb) | fMRI participants' post-scan survey | **same** people as transcripts/brain | individual | end-state only (1 final impression) | 35 |

 A has the temporal resolution B lacks. B has the person
resolution A lacks

In [6]:
import pandas as pd, numpy as np, scipy.io as sio
from pathlib import Path

# --- Track B ground truth: fMRI participants' OWN post-scan survey --------------
# One .mat per fMRI participant (same IDs as the transcripts). Layout is data1..data6,
# each (n_items x 4 characters). NO run dimension -- a single end-state administration.
# NOTE: this is the fMRI participants' POST-SCAN survey (data1..data6 per participant),
# a DIFFERENT dataset from Track A's run-resolved data/charsurvey (block1..block10).
# Point this at the post-scan folder; candidates are tried in order, first that exists wins.
SURVEY_DIR_CANDIDATES = [Path("/Users/rheamadhogarhia/Desktop/CABLAB RESEARCH/independent ra work/part_4_summer_char_profiles/characterprofilesynchronygit/data/ALL DATA OG SHARE socialaha-collab/socialaha-fMRI/charactersurvey")]
SURVEY_DIR = next((p for p in SURVEY_DIR_CANDIDATES if p.exists()), SURVEY_DIR_CANDIDATES[0])
CHAR_COLS  = ["jack", "kate", "randall", "kevin"]   # column order (survey sheet p.1)

# 35 item labels in block order, from charactersurvey.pdf (NO labels.mat exists for these).
BLOCK_LABELS = {
 "data1": ["warm and kind","intelligent","agreeable","extraverted","impulsive",
           "emotionally stable","open-minded","trustworthy","competent"],          # 9 traits (p.2)
 "data2": ["rational behavior","positive emotion","good relationship"],            # 3 statements (p.2)
 "data3": ["empathize","understand","like","similar"],                             # 4 thoughts (p.2)
 "data4": ["friendly","assertive","reserved","self-disciplined","optimistic",
           "humorous","sincere and honest","self-centered","determined",
           "caring and supportive","ambitious"],                                   # 11 traits (p.3)
 "data5": ["good job","content","big influence"],                                  # 3 statements (p.3)
 "data6": ["want to be character","want to be friends","care what happens",
           "annoying","attractive"],                                               # 5 thoughts (p.3)
}
ALL_ITEMS = [lab for blk in ["data1","data2","data3","data4","data5","data6"] for lab in BLOCK_LABELS[blk]]

# Valence coding + construct definitions come from the shared module (same source Track A uses),
# so the two tracks cannot drift. Track B uses the FULL 35-item survey; scope=ITEMS_16 gives the
# Track-A-matched composite.
from behavioral_constructs import (valence_composite, stance_composite, ITEMS_16,
                                   CHAR_EMOTION_ITEM, LIKE_ITEM,
                                   CHAR_POSITIVE, CHAR_NEGATIVE, STANCE_POSITIVE, STANCE_NEGATIVE)

def group_from_id(pid):
    digits=[c for c in str(pid) if c.isdigit()]
    return int(digits[0]) if digits else np.nan

print("35 items across 6 blocks; character-valence uses", len(CHAR_POSITIVE)+len(CHAR_NEGATIVE),
      "items (full);", len([c for c in CHAR_POSITIVE+CHAR_NEGATIVE if c in ITEMS_16]), "in the Track-A-matched 16")


35 items across 6 blocks; character-valence uses 22 items (full); 11 in the Track-A-matched 16


In [7]:
# STAGE 1: load each participant's 35-item x 4-character end-state survey into long form.
# One row per (participant, character); items become columns. No run dimension.
# ROBUST: the post-scan survey stores blocks data1..data6. If SURVEY_DIR instead holds the Track-A
# run-resolved survey (block1..block10), we DETECT that and skip gracefully rather than KeyError.
def _postscan_keys_ok(m):
    return all(f"data{i}" in m for i in range(1, 7))

def load_survey(path):
    pid = Path(path).stem
    m = sio.loadmat(path)
    if not _postscan_keys_ok(m):
        have = [k for k in m if not k.startswith("__")]
        raise ValueError(f"{pid}: expected post-scan blocks data1..data6, found {have}. "
                         "This looks like the Track-A run-resolved survey (block1..block10), NOT the "
                         "fMRI post-scan survey. Point SURVEY_DIR at the post-scan folder.")
    mat = np.vstack([np.asarray(m[blk], dtype=float) for blk in
                     ["data1","data2","data3","data4","data5","data6"]])
    assert mat.shape == (35, 4), f"{pid}: expected (35,4), got {mat.shape}"
    rows = []
    for ci, ch in enumerate(CHAR_COLS):
        rec = {"Participant": pid, "group": group_from_id(pid), "Character": ch}
        for ri, lab in enumerate(ALL_ITEMS):
            rec[lab] = mat[ri, ci]
        rows.append(pd.DataFrame([rec]))
    return pd.concat(rows, ignore_index=True)

def _is_postscan_file(p):
    try:
        return _postscan_keys_ok(sio.loadmat(p))
    except Exception:
        return False

_cands = [p for p in sorted(SURVEY_DIR.glob("*.mat")) if p.name != "labels.mat"] if SURVEY_DIR.exists() else []
files  = [p for p in _cands if _is_postscan_file(p)]
if files:
    surv = pd.concat([load_survey(f) for f in files], ignore_index=True)
    print("participants:", surv["Participant"].nunique(), "| rows:", len(surv), "(= participants x 4 chars)")
elif _cands:
    surv = None
    print(f"SURVEY_DIR={SURVEY_DIR} has .mat files but NONE are post-scan layout (data1..data6).")
    print("  -> that folder looks like the Track-A survey (block1..block10). Point SURVEY_DIR at the post-scan folder.")
else:
    surv = None
    print(f"SURVEY_DIR={SURVEY_DIR} not populated yet -- set it to the post-scan survey folder and re-run.")

participants: 36 | rows: 144 (= participants x 4 chars)


In [8]:
# STAGE 2: build the final-impression constructs per (participant, character).
# Reverse-code negatives (8 - x); neutral items are EXCLUDED from valence composites.
def build_constructs(surv):
    g=surv.copy()
    g["t_char_emotion"]        = g[CHAR_EMOTION_ITEM]                        # single, matches Track A
    g["t_like"]                = g[LIKE_ITEM]                                # single, matches Track A
    g["t_char_valence_16"]     = valence_composite(g, scope=ITEMS_16)        # 16-item, A-comparable
    g["t_char_valence_full"]   = valence_composite(g, scope=None)            # full 35-item version
    g["t_participant_stance"]  = stance_composite(g, scope=None)             # participant attitude
    keep=["Participant","group","Character","t_char_emotion","t_like",
          "t_char_valence_16","t_char_valence_full","t_participant_stance"]
    return g[keep]

if surv is not None:
    constructs=build_constructs(surv)
    Path("results/step2").mkdir(parents=True, exist_ok=True)
    constructs.to_csv("results/step2/02__final_impression_constructs.csv", index=False)
    print("saved constructs:", constructs.shape, "-> results/step2/02__final_impression_constructs.csv")
else:
    constructs=None


saved constructs: (144, 8) -> results/step2/02__final_impression_constructs.csv


In [9]:
# STAGE 3: end-state SENTIMENT per (participant, character) from the transcripts.
# The survey is a single final impression, so summarise each person's run trajectory two ways:
#   val_mean  = mean valence across the 10 runs      (overall expressed sentiment)
#   val_final = valence at the last available run     (closest match to a post-scan impression)
SENT_PROFILE = "results/profiles/00__profiles_Twitter_RoB.csv"   # Participant,Run,Character,pos,neg
prof=pd.read_csv(SENT_PROFILE)
prof["valence"]=prof["Twitter_RoB_pos"]-prof["Twitter_RoB_neg"]
# normalise IDs so survey "s1001" and profile "sub-1001" match on the numeric id
def normid(x):
    d="".join(c for c in str(x) if c.isdigit()); return d
prof["pid"]=prof["Participant"].map(normid)
val_mean=prof.groupby(["pid","Character"])["valence"].mean().rename("val_mean")
last=prof.sort_values("Run").groupby(["pid","Character"]).tail(1)
val_final=last.set_index(["pid","Character"])["valence"].rename("val_final")
sent=pd.concat([val_mean,val_final],axis=1).reset_index()
print("sentiment end-state rows:",len(sent),"| participants:",sent["pid"].nunique())


sentiment end-state rows: 128 | participants: 32


In [10]:
# STAGE 4: individual-level validation -- merge each person's survey construct with their OWN sentiment.
from scipy.stats import spearmanr, pearsonr
from scipy.stats import rankdata

def clustered_boot_ci(d, xcol, ycol, nboot=5000, seed=0):
    # cluster (participant) bootstrap; numpy-only for speed
    rng=np.random.default_rng(seed)
    groups=[g[[xcol,ycol]].to_numpy() for _,g in d.groupby("pid")]
    pids=np.arange(len(groups)); rs=[]
    for _ in range(nboot):
        samp=np.concatenate([groups[i] for i in rng.choice(pids,len(pids),replace=True)])
        x,y=samp[:,0],samp[:,1]
        if len(np.unique(x))>2 and len(np.unique(y))>2:
            rx,ry=rankdata(x),rankdata(y)
            rs.append(np.corrcoef(rx,ry)[0,1])
    return (np.nanpercentile(rs,2.5), np.nanpercentile(rs,97.5)) if rs else (np.nan,np.nan)

def validate(constructs, sent, nboot=5000, save=True):
    c=constructs.copy(); c["pid"]=c["Participant"].map(normid)
    m=c.merge(sent, on=["pid","Character"], how="inner")
    print("individual-level overlap: %d participants x characters = %d rows\n"
          % (m["pid"].nunique(), len(m)))
    targets=["t_char_emotion","t_char_valence_16","t_char_valence_full","t_like","t_participant_stance"]
    rows=[]
    for t in targets:
        for scol in ["val_mean","val_final"]:
            d=m.dropna(subset=[t,scol])
            if len(d)<8: continue
            rho=spearmanr(d[scol],d[t]).correlation
            r  =pearsonr(d[scol],d[t])[0]
            lo,hi=clustered_boot_ci(d,scol,t,nboot=nboot)
            rows.append({"construct":t,"sentiment":scol,"n":len(d),
                         "spearman":round(rho,3),"pearson":round(r,3),
                         "ci_lo":round(lo,3),"ci_hi":round(hi,3)})
    res=pd.DataFrame(rows)
    if save:
        res.to_csv("results/step2/02__validation.csv", index=False)
    return res

if constructs is not None:
    result=validate(constructs, sent)
    print(result.to_string(index=False))
else:
    print("Populate SURVEY_DIR and re-run to produce the Track B validation table.")


individual-level overlap: 31 participants x characters = 124 rows

           construct sentiment   n  spearman  pearson  ci_lo  ci_hi
      t_char_emotion  val_mean 124     0.416    0.418  0.258  0.554
      t_char_emotion val_final 124     0.165    0.179 -0.047  0.359
   t_char_valence_16  val_mean 124     0.559    0.564  0.427  0.680
   t_char_valence_16 val_final 124     0.185    0.153 -0.066  0.414
 t_char_valence_full  val_mean 124     0.574    0.593  0.443  0.694
 t_char_valence_full val_final 124     0.144    0.117 -0.107  0.369
              t_like  val_mean 124     0.385    0.392  0.181  0.566
              t_like val_final 124     0.178    0.147 -0.027  0.377
t_participant_stance  val_mean 124     0.370    0.393  0.159  0.560
t_participant_stance val_final 124     0.152    0.143 -0.038  0.334


## Reading Track B

- **`t_char_valence_16`** is the apples-to-apples comparison with Track A: same character items, now at
  the individual level against each viewer's own final impression.
- **`t_char_valence_full`** uses all 35 survey items (`self-centered` and `impulsive` reverse-coded), the
  richer composite the post-scan survey uniquely allows.
- **`t_participant_stance`** is the viewer-attitude construct (like / empathize / want-to-be-friends /
  attractive, `annoying` reversed) -- the individual-level analog of Track A's participant target.
- `val_mean` vs `val_final`: match with overall expressed sentiment vs the last run.

End-state only -- cannot speak to run-by-run change (that is Track A's job). Report Track A
and Track B side by side; do not pool them (different cohorts, resolutions, item sets).

**Result (real run -- 36 fMRI participants loaded, 32 overlap with transcripts x 4 chars = 128 rows).**
Overall expressed sentiment (`val_mean`), Spearman vs each construct:

| construct | rho (val_mean) | rho (val_final) |
|---|---|---|
| `t_char_valence_full` (35-item) | **0.574** | 0.144 |
| `t_char_valence_16` (A-comparable) | **0.559** | 0.185 |
| `t_char_emotion` (positive emotion) | 0.416 | 0.165 |
| `t_participant_stance` (like/empathize/...) | 0.370 | 0.152 |
| `t_like` (single) | 0.385 | 0.178 |

at the individual, same-cohort, end-state level the sentiment measure lines up best with the
**character-valence composite** (0.574 / 0.559) -- the same character-affect story as Track A. Two things
differ from Track A: (1) **participant stance / `like` are modestly *positive* here (~0.37-0.39)**, not
the ~0 they were at Track A's group level; and (2) **`val_mean` beats `val_final` everywhere** -- a
person's *average* expressed sentiment across the 10 runs matches their end-state survey better than
their last run does.

> [!note] Values updated 2026-07-21 to the cleaned-transcript rerun (Jin's transcripts); validity rose ~0.03–0.05 vs the July estimates.


## ASK HAYOUNG!!
- **ASK HAYOUNG!!** Confirm survey scope: run-resolved SONA = the 16-item page; the 35-item survey is the fMRI post-scan only (no run-resolved 35-item exists?).
- **ASK HAYOUNG!!** Confirm the character column order in the post-scan `.mat` blocks (Jack/Kate/Randall/Kevin).

CONFUSED ABOUT THIS NOTEBOOK AND WHETHERE IT MADE AN IMPACT ON ANYTHING DOWNSTREAM
